# Hate Speech Detection on Civil Comments
## Final hierarchical CNN + Bi-LSTM experiment

**Executable research notebook — scientific initiation final delivery**

This is the canonical notebook for reproducing the final experiment. Run it **top-to-bottom in a clean Google Colab runtime**, preferably with GPU, and publish the executed copy **with outputs preserved**.

**Research design:** Stage 1 routes comments using toxicity; Stage 2 predicts `obscene`, `threat`, `insult`, `identity_attack`, and `sexual_explicit`. Ground-truth routing is `toxicity >= 0.4`; prediction thresholds are selected only on inner validation. The outer folds are evaluation-only.

The implementation lives in `src/` and `scripts/`; this notebook records provenance, validates the repository, runs the full experiment, parses the observed metrics, and produces the final tables/plots.

## 1. Reproduction setup

The notebook uses a clean clone of `main`, avoiding hidden notebook state and nested clones. The exact Git commit and a SHA-256 fingerprint of the scientific files are recorded before training.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

URL="https://github.com/Umbura/Hatespeech_Detection_Civil_Comments_NLP_Obsolete.git"
BASE=Path("/content") if Path("/content").exists() else Path.cwd()
REPO=BASE/"hatespeech-final-reproduction"

if REPO.exists():
    if not (REPO/".git").exists(): raise RuntimeError(f"{REPO} is not a Git clone.")
    if subprocess.check_output(["git","status","--porcelain"],cwd=REPO,text=True).strip():
        raise RuntimeError("Existing reproduction clone has local changes.")
    subprocess.run(["git","fetch","origin","--prune"],cwd=REPO,check=True)
    subprocess.run(["git","switch","main"],cwd=REPO,check=True)
    subprocess.run(["git","pull","--ff-only","origin","main"],cwd=REPO,check=True)
else:
    subprocess.run(["git","clone","--branch","main","--single-branch",URL,str(REPO)],check=True)

os.chdir(REPO)
COMMIT=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
print("Repository:",REPO)
print("Commit:",COMMIT)

## 2. Software environment

In [ ]:
PACKAGES=[
 "tensorflow==2.20.0","numpy==2.0.2","pandas==2.2.3","scikit-learn==1.6.1",
 "imbalanced-learn==0.14.2","datasets==5.0.0","iterative-stratification==0.1.9"
]
subprocess.run([sys.executable,"-m","pip","install","-q",*PACKAGES],check=True)
subprocess.run([sys.executable,"-m","pip","check"],check=True)

import hashlib, importlib.metadata as meta, json, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd, sklearn, tensorflow as tf

FILES=[
 "scripts/run_hierarchical_cv.py","src/hate_speech_detection/cv_pipeline.py",
 "src/hate_speech_detection/hierarchical_splits.py","src/hate_speech_detection/target_strategy.py",
 "src/hate_speech_detection/threshold_selection.py","requirements.txt"
]
def sha(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""): h.update(chunk)
    return h.hexdigest()

hashes={p:sha(REPO/p) for p in FILES}
manifest=hashlib.sha256("\n".join(f"{p}\0{hashes[p]}" for p in sorted(hashes)).encode()).hexdigest()
gpus=tf.config.list_physical_devices("GPU")
hardware="CPU"
if gpus and shutil.which("nvidia-smi"):
    q=subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True)
    hardware=q.stdout.strip() or str(gpus)

runtime={
 "timestamp_utc":datetime.now(timezone.utc).isoformat(),"git_commit":COMMIT,
 "scientific_manifest_sha256":manifest,"python":platform.python_version(),
 "tensorflow":tf.__version__,"numpy":np.__version__,"pandas":pd.__version__,
 "scikit_learn":sklearn.__version__,"datasets":meta.version("datasets"),"hardware":hardware
}
display(pd.DataFrame(runtime.items(),columns=["Item","Value"]))

## 3. Repository validation

Before training, the notebook checks dependency compatibility, compiles the source/scripts, and runs the repository regression suite.

In [ ]:
for cmd in [
 [sys.executable,"-m","pip","check"],
 [sys.executable,"-m","compileall","-q","src","scripts"],
 [sys.executable,"-m","unittest","discover","-s","tests","-p","test_*.py","-v"],
]:
    print("$"," ".join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)
print("\nRepository validation: PASSED")

## 4. Documented reference

Reference metrics are loaded from `results/final_metrics.json` rather than duplicated in the notebook. The primary documented end-to-end Macro F1 is **0.4412**; **0.4427** is the recorded replication.

In [ ]:
ref=json.loads((REPO/"results"/"final_metrics.json").read_text())
primary,rep=ref["primary_run"],ref["replication_run"]
display(pd.DataFrame([
 ("Stage 1 F1 (nested)",primary["stage1_nested"]["f1"]),
 ("Stage 1 recall (nested)",primary["stage1_nested"]["recall"]),
 ("Stage 1 PR-AUC / AP",primary["stage1_nested"]["average_precision"]),
 ("Stage 1 ROC-AUC",primary["stage1_nested"]["roc_auc"]),
 ("Stage 2 oracle Macro F1",primary["stage2_oracle_nested_macro_f1"]),
 ("End-to-end Macro F1 (fixed)",primary["end_to_end_fixed_macro_f1"]),
 ("End-to-end Macro F1 (nested)",primary["end_to_end_nested_macro_f1"]),
 ("Replication end-to-end Macro F1",rep["end_to_end_nested_macro_f1"]),
],columns=["Metric","Reference"]).style.format({"Reference":"{:.4f}"}))

## 5. Full experiment

Runs the canonical command on the full Civil Comments training split:

`python scripts/run_hierarchical_cv.py --n-splits 2 --epochs 5`

The full log is saved outside the Git working tree. CPU is supported, but GPU is strongly recommended.

In [ ]:
import re,time
RUN=BASE/"hate_speech_final_run"; RUN.mkdir(exist_ok=True)
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
log_path=RUN/f"full_{stamp}.log"
cmd=[sys.executable,"scripts/run_hierarchical_cv.py","--n-splits","2","--epochs","5"]

started=time.perf_counter(); lines=[]
p=subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                   text=True,bufsize=1,env={**os.environ,"PYTHONUNBUFFERED":"1"})
assert p.stdout
show=("Fold ","best inner-validation epoch:","Selected inner thresholds:","--- Ground-truth",
      "--- Threshold selection","--- Stage 1","--- Stage 2","--- End-to-end",
      "Macro F1:","Accuracy:","Precision:","Recall:","Routing rate:","PR-AUC","ROC-AUC:",
      "obscene F1:","threat F1:","insult F1:","identity_attack F1:","sexual_explicit F1:",
      "Routed samples:","any Stage 2 label:")
with log_path.open("w",encoding="utf-8") as f:
    for line in p.stdout:
        lines.append(line); f.write(line); f.flush()
        if any(x in line for x in show): print(line.rstrip(),flush=True)
rc=p.wait(); elapsed=time.perf_counter()-started
if rc: raise RuntimeError(f"Full run failed ({rc}). See {log_path}")
full_log="".join(lines)
print(f"\nFull experiment: PASSED — {elapsed/60:.1f} min")

## 6. Parse the observed metrics

In [ ]:
def sec(title):
    m=re.search(rf"--- {re.escape(title)} ---\s*(.*?)(?=\n--- |\Z)",full_log,re.S)
    if not m: raise ValueError(f"Missing section: {title}")
    return m.group(1)
def s1(title):
    d={}
    for line in sec(title).splitlines():
        if ": " in line:
            k,v=line.split(": ",1)
            try:d[k]=float(v.split(" | ")[0])
            except ValueError:pass
    return d
LABELS=["obscene","threat","insult","identity_attack","sexual_explicit"]
def s2(title):
    t=sec(title); macro=float(re.search(r"^Macro F1:\s*([0-9.]+)",t,re.M).group(1))
    return {"macro_f1":macro,"per_label":{
        x:float(re.search(rf"^{x} F1:\s*([0-9.]+)",t,re.M).group(1)) for x in LABELS}}

s1f=s1("Stage 1 toxicity gate — fixed prediction threshold 0.40")
s1n=s1("Stage 1 toxicity gate — nested tuned routing threshold")
s2f=s2("Stage 2 oracle — fixed label threshold 0.50")
s2n=s2("Stage 2 oracle — nested tuned label thresholds")
e2ef=s2("End-to-end — fixed 0.40 routing / 0.50 labels")
e2en=s2("End-to-end — nested tuned thresholds")

tt=sec("Threshold selection summary")
folds=[dict(fold=int(a),routing_threshold=float(b),inner_e2e_macro_f1=float(c),
            inner_gate_recall=float(d),inner_routing_rate=float(e))
       for a,b,c,d,e in re.findall(
        r"Fold (\d+): routing=([0-9.]+), inner end-to-end Macro F1=([0-9.]+), "
        r"inner gate recall=([0-9.]+), inner routing rate=([0-9.]+)",tt)]
ct=sec("Ground-truth gate coverage analysis")
r=re.search(r"Routed samples: (\d+)/(\d+) using toxicity >= ([0-9.]+)",ct)
a=re.search(r"any Stage 2 label: positives=(\d+), missed=(\d+) \(([0-9.]+)%\)",ct)
assert r and a
coverage={"routed":int(r.group(1)),"total":int(r.group(2)),"gate":float(r.group(3)),
          "positive":int(a.group(1)),"missed":int(a.group(2))}
print("Result parsing: PASSED")

## 7. Final results

In [ ]:
stage1=pd.DataFrame({
 "Fixed 0.40":[s1f[k] for k in ["Accuracy","Precision","Recall","F1","Routing rate","PR-AUC (average precision)","ROC-AUC"]],
 "Nested":[s1n[k] for k in ["Accuracy","Precision","Recall","F1","Routing rate","PR-AUC (average precision)","ROC-AUC"]],
},index=["Accuracy","Precision","Recall","F1","Routing rate","PR-AUC / AP","ROC-AUC"])
display(stage1.style.format("{:.4f}"))

oracle=pd.DataFrame({"Fixed 0.50":[s2f["per_label"][x] for x in LABELS],
                     "Nested":[s2n["per_label"][x] for x in LABELS]},index=LABELS)
oracle.loc["Macro F1"]=[s2f["macro_f1"],s2n["macro_f1"]]
display(oracle.style.format("{:.4f}"))

e2e=pd.DataFrame({"Fixed":[e2ef["per_label"][x] for x in LABELS],
                  "Nested":[e2en["per_label"][x] for x in LABELS]},index=LABELS)
e2e.loc["Macro F1"]=[e2ef["macro_f1"],e2en["macro_f1"]]
display(e2e.style.format("{:.4f}"))

gain=e2en["macro_f1"]-e2ef["macro_f1"]
print(f"End-to-end absolute gain: {gain:+.4f}")
print(f"End-to-end relative gain: {gain/e2ef['macro_f1']*100:+.1f}%")

## 8. Thresholds, gate coverage, and figures

In [ ]:
display(pd.DataFrame(folds).style.format(precision=4))
display(pd.DataFrame([
 ("Total samples",coverage["total"]),("Routed samples",coverage["routed"]),
 ("Routing share",coverage["routed"]/coverage["total"]),
 ("Any Stage 2 positive",coverage["positive"]),("Any-positive missed",coverage["missed"]),
 ("Any-positive coverage",1-coverage["missed"]/coverage["positive"]),
],columns=["Metric","Value"]))

import matplotlib.pyplot as plt
summary=pd.DataFrame({
 "Fixed":[s1f["F1"],s2f["macro_f1"],e2ef["macro_f1"]],
 "Nested":[s1n["F1"],s2n["macro_f1"],e2en["macro_f1"]],
},index=["Stage 1 F1","Stage 2 oracle Macro F1","End-to-end Macro F1"])
ax=summary.plot(kind="bar",figsize=(9,5)); ax.set_ylim(0,.8); ax.set_ylabel("F1")
ax.set_title("Fixed vs. nested thresholds"); ax.grid(axis="y",alpha=.25)
plt.xticks(rotation=15,ha="right"); plt.tight_layout(); plt.show()

ax=e2e.drop(index="Macro F1").plot(kind="bar",figsize=(10,5)); ax.set_ylim(0,.85)
ax.set_ylabel("F1"); ax.set_title("End-to-end F1 by label"); ax.grid(axis="y",alpha=.25)
plt.xticks(rotation=20,ha="right"); plt.tight_layout(); plt.show()

## 9. Reproduction vs. documented benchmark

This table reports the current run separately. It does not automatically replace the primary benchmark with a slightly larger stochastic run.

In [ ]:
p=primary["end_to_end_nested_macro_f1"]; q=rep["end_to_end_nested_macro_f1"]; c=e2en["macro_f1"]
display(pd.DataFrame([
 ("Primary documented benchmark",p,0.0),
 ("Documented replication",q,q-p),
 ("This notebook run",c,c-p),
],columns=["Run","End-to-end Macro F1","Δ vs. primary"]).style.format({
 "End-to-end Macro F1":"{:.4f}","Δ vs. primary":"{:+.4f}"
}))
print(f"Stage 1 nested F1       : {s1n['F1']:.4f}")
print(f"Stage 1 nested recall   : {s1n['Recall']:.4f}")
print(f"Stage 2 oracle Macro F1 : {s2n['macro_f1']:.4f}")
print(f"End-to-end Macro F1     : {e2en['macro_f1']:.4f}")
print(f"Oracle → E2E gap        : {s2n['macro_f1']-e2en['macro_f1']:.4f}")

## 10. Reproduction artifact and limitations

A machine-readable summary is saved outside the clone. Known limitations remain explicit: two outer folds; early overfitting pressure; routing-error propagation; no completed subgroup fairness/robustness study; no frozen official-test benchmark; no SOTA or production-readiness claim. Seeds are set by the runner, but byte-identical results are not guaranteed across different hardware/runtime stacks.

In [ ]:
artifact={
 "status":"completed","command":cmd,"elapsed_seconds":elapsed,"runtime":runtime,
 "coverage":coverage,"folds":folds,
 "observed":{"stage1_fixed":s1f,"stage1_nested":s1n,
             "stage2_oracle_fixed":s2f,"stage2_oracle_nested":s2n,
             "end_to_end_fixed":e2ef,"end_to_end_nested":e2en}
}
artifact_path=RUN/f"reproduction_{stamp}.json"
artifact_path.write_text(json.dumps(artifact,indent=2,ensure_ascii=False),encoding="utf-8")
print("Structured artifact:",artifact_path)
print("Full log:",log_path)
print("Scientific manifest SHA-256:",manifest)

## Publication checklist

Before committing this notebook to GitHub:

- `Repository validation: PASSED`;
- `Full experiment: PASSED`;
- final Stage 1 / Stage 2 oracle / end-to-end tables are visible;
- thresholds and gate coverage are visible;
- plots rendered;
- commit, software versions and hardware are recorded;
- **outputs are preserved**.

Publish the executed notebook at `notebooks/final/HateSpeech_Final_Hierarchical.ipynb`. The historical notebook remains separate for traceability only.